In [0]:
# Create and setup parameter
import json
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
# define input widget
dbutils.widgets.text("SchemaValidationJSON", "", "")
schema_validation_json = dbutils.widgets.get("SchemaValidationJSON")

In [0]:
# parse the JSON string directly into a dataframe
file_id_df = spark.read.json((spark.sparkContext.parallelize([schema_validation_json])))

In [0]:
# Explode and flatten the configuration metadata fields
exploded_file_ids = file_id_df.select(F.explode("FileIds").alias("col")).select(
    F.col("col.FileID").alias("FileID"),
    F.col("col.FileName").alias("FileName"),
    F.col("col.RegistrationDatetime").alias("RegistrationDatetime"),
    F.col("col.FullCurrentPath").alias("FullCurrentPath"),
    F.col("col.CurrentContainer").alias("CurrentContainer"),
    F.col("col.CurrentFolderPath").alias("CurrentFolderPath"),
    F.col("col.ValidatedContainer").alias("ValidatedContainer"),
    F.col("col.ValidatedFolderPath").alias("ValidatedFolderPath"),
    F.col("col.FullValidatedPath").alias("FullValidatedPath"),
    F.col("col.ValidationFileContainer").alias("ValidationFileContainer"),
    F.col("col.ValidationFileFolderPath").alias("ValidationFileFolderPath"),
    F.col("col.ValidationFileName").alias("ValidationFileName"),
    F.col("col.ColumnDelimiter").alias("ColumnDelimiter"),
    F.col("col.ConversionType").alias("ConversionType"),
    F.col("col.HasHeaders").alias("HasHeaders"),
    F.col("col.IgnoreHeader").alias("IgnoreHeader"),
    F.col("col.TextQualifier").alias("TextQualifier")
)

In [0]:
# Orchestrate loop and execute level 2 validations
output_records = []

# collect rows safely to iterate on the driver node
for row in exploded_file_ids.collect():
    config = row.asDict()
    file_name = config["FileName"]

    # Traget conversion formatting for healthcare EDI 837 claim files
    if config["ConversionType"] == "837" and "." in file_name:
        file_name = file_name[0:file_name.rfind(".")] + ".csv"
    
        results_str = ""
    try:
        # Run the downstream Level2Validations notebook matching Scala parameters
        results_str = dbutils.notebook.run(
            "Level2Validations",
            0,
            {
                "FileID": str(config["FileID"]),
                "FileName": str(file_name),
                "CurrentContainer": str(config["CurrentContainer"]),
                "CurrentFolderPath": str(config["CurrentFolderPath"]),
                "ValidationContainer": str(config["ValidationFileContainer"]),
                "ValidationFileFolderPath": str(config["ValidationFileFolderPath"]),
                "ValidationFileName": str(config["ValidationFileName"]),
                "ValidatedFolderPath": str(config["FullValidatedPath"]),
                "HasHeaders": str(config["HasHeaders"]),
                "IgnoreHeader": str(config["IgnoreHeader"]),
                "Delimiter": str(config["ColumnDelimiter"]),
                "TextQualifier": str(config["TextQualifier"])
            }
        )
    except Exception as e:
        # Handle workflow errors safely without crashing the iteration loop
        error_payload = {
            "CurrentJobId": "",
            "RecordCount": "0",
            "ErrorCount": "0",
            "ErrorPathSchema": "",
            "Status": "FAILED",
            "ErrorMessage": str(e).replace('"', '')
        }
        results_str = json.dumps(error_payload)

    # Parse notebook return payload strings safely back into dictionaries
    try:
        returned_data = json.loads(results_str)
        # Handle if the notebook returns an array wrapper
        if isinstance(returned_data, list) and len(returned_data) > 0:
            returned_data = returned_data[0]
    except Exception:
        returned_data = {
            "CurrentJobId": "", "RecordCount": "0", "ErrorCount": "0",
            "ErrorPathSchema": "", "Status": "FAILED", "ErrorMessage": str(results_str)
        }

    # Consolidate target input configurations with rule execution metrics
    combined_record = {
        "FileID": config["FileID"],
        "FullCurrentPath": config["FullCurrentPath"],
        "FileName": config["FileName"],
        "CurrentContainer": config["CurrentContainer"],
        "CurrentFolderPath": config["CurrentFolderPath"],
        "ValidatedContainer": config["ValidatedContainer"],
        "ValidatedFolderPath": config["ValidatedFolderPath"],
        "FullValidatedPath": config["FullValidatedPath"],
        "ConversionType": config["ConversionType"],
        "RegistrationDatetime": config["RegistrationDatetime"],
        "CurrentJobId": returned_data.get("CurrentJobId", ""),
        "RecordCount": returned_data.get("RecordCount", "0"),
        "ErrorCount": returned_data.get("ErrorCount", "0"),
        "ErrorPathSchema": returned_data.get("ErrorPathSchema", ""),
        "Status": returned_data.get("Status", ""),
        "ErrorMessage": returned_data.get("ErrorMessage", "")
    }
    output_records.append(combined_record)

# Exit with Consolidated JSON Summary Engine Data
# Terminate and return clean structured JSON string arrays
dbutils.notebook.exit(json.dumps(output_records))